# Module 6: Monte Carlo Simulation & Stress Testing

**QuantVerse** — Quantitative Portfolio Intelligence System

---

## Objectives

Forward-looking portfolio analysis using simulation and stress testing:

1. **Monte Carlo Simulation** — 10,000 paths × 252 days (1-year forward)
   - Multivariate Normal
   - Multivariate Student-t (fat tails)
   - Block Bootstrap (preserves autocorrelation)
   - GARCH-filtered Bootstrap (volatility clustering)
2. **Probability Analysis** — P(gain), P(loss >X%), confidence intervals
3. **Historical Stress Replay** — COVID, rate hikes, crypto winter, etc.
4. **Hypothetical Stress Scenarios** — equity crash, stagflation, rate shock
5. **Reverse Stress Testing** — what breaks the portfolio?

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, logging, sys, os, json

sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_palette('husl')
print('Setup complete.')

In [ ]:
# Load data
data_dir = '../data/processed'
daily_returns = pd.read_parquet(f'{data_dir}/returns_daily.parquet')
clean_prices = pd.read_parquet(f'{data_dir}/prices_clean.parquet')

with open(f'{data_dir}/asset_class_map.json', 'r') as f:
    class_map = json.load(f)

# Investable assets
signal_tickers = [t for t, c in class_map.items() if c == 'signals']
investable = [t for t in daily_returns.columns if t not in signal_tickers]
returns = daily_returns[investable].dropna()

# Load portfolio weights
weights_path = f'{data_dir}/portfolio_weights.parquet'
if os.path.exists(weights_path):
    all_weights = pd.read_parquet(weights_path)
else:
    all_weights = pd.DataFrame({'Equal Weight': pd.Series(1/len(investable), index=investable)})

# Primary strategy
primary = 'Max Sharpe' if 'Max Sharpe' in all_weights.columns else all_weights.columns[0]
w_primary = all_weights[primary]

print(f'Assets: {len(investable)}, Obs: {len(returns)}')
print(f'Primary strategy: {primary}')
print(f'All strategies: {list(all_weights.columns)}')

## 1. Monte Carlo Simulation — All Methods

In [ ]:
from project.simulation import MonteCarloSimulator, ScenarioAnalyzer

mc = MonteCarloSimulator(
    returns=returns,
    weights=w_primary,
    horizon_days=252,
    n_sims=10_000,
    seed=42
)

all_sims = mc.simulate_all(student_df=5, block_size=21)
print(f'\nCompleted {len(all_sims)} simulation methods')

In [ ]:
# Summary table across methods
summary = ScenarioAnalyzer.summary_table(all_sims)
print('Monte Carlo Summary — 12-Month Forward Projection')
print('=' * 110)
print(summary.round(2).to_string())

## 2. Probability Cone — Primary Strategy

In [ ]:
# Normal simulation cone
fig = ScenarioAnalyzer.plot_probability_cone(all_sims['Normal'], n_sample_paths=100)
plt.show()

In [ ]:
# Student-t simulation cone (fatter tails)
fig = ScenarioAnalyzer.plot_probability_cone(all_sims['Student-t'], n_sample_paths=100)
plt.show()

## 3. Terminal Return Distribution Comparison

In [ ]:
fig = ScenarioAnalyzer.plot_method_comparison(all_sims)
plt.show()

In [ ]:
# Terminal distribution deep dive — Normal
fig = ScenarioAnalyzer.plot_terminal_distribution(all_sims['Normal'])
plt.show()

In [ ]:
# Terminal distribution — Student-t
fig = ScenarioAnalyzer.plot_terminal_distribution(all_sims['Student-t'])
plt.show()

## 4. Maximum Drawdown Distribution

In [ ]:
fig = ScenarioAnalyzer.plot_drawdown_distribution(all_sims['Bootstrap'])
plt.show()

In [ ]:
# Drawdown distribution comparison
fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
colors = ['steelblue', 'coral', 'seagreen', 'mediumpurple']

for idx, (name, result) in enumerate(all_sims.items()):
    mdd = result['max_drawdowns'] * 100
    axes[idx].hist(mdd, bins=60, density=True, alpha=0.6, color=colors[idx], edgecolor='white')
    axes[idx].axvline(np.median(mdd), color='black', lw=2)
    axes[idx].set_title(f"{name}\nMedian DD: {np.median(mdd):.1f}%", fontweight='bold', fontsize=10)
    axes[idx].set_xlabel('Max Drawdown (%)')

axes[0].set_ylabel('Density')
plt.suptitle('Maximum Drawdown Distribution by Simulation Method',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Multi-Strategy Monte Carlo Comparison

In [ ]:
# Run bootstrap simulation for all strategies
strategy_sims = {}
strategies_to_compare = list(all_weights.columns)[:6]

for strat in strategies_to_compare:
    mc_s = MonteCarloSimulator(returns, all_weights[strat], horizon_days=252, n_sims=5000, seed=42)
    strategy_sims[strat] = mc_s.simulate_bootstrap(block_size=21)

# Comparison table
strat_summary = ScenarioAnalyzer.summary_table(strategy_sims)
print('Strategy Comparison — Bootstrap MC (5,000 sims, 1yr horizon)')
print('=' * 110)
print(strat_summary[['Median_Return_%', 'VaR_5%', 'CVaR_5%', 'P(Positive)_%',
                      'P(Loss>10%)_%', 'P(Gain>20%)_%', 'Avg_MaxDD_%']].round(2).to_string())

In [ ]:
# Violin plot comparison
fig, ax = plt.subplots(figsize=(14, 7))

data_for_violin = []
labels = []
for strat, result in strategy_sims.items():
    data_for_violin.append(result['terminal_returns'] * 100)
    labels.append(strat)

parts = ax.violinplot(data_for_violin, showmedians=True, showextrema=False)
for i, pc in enumerate(parts['bodies']):
    pc.set_alpha(0.6)

ax.set_xticks(range(1, len(labels) + 1))
ax.set_xticklabels(labels, rotation=25, ha='right')
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
ax.set_ylabel('12-Month Return (%)')
ax.set_title('Projected Return Distribution by Strategy (Bootstrap MC)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Historical Stress Testing

In [ ]:
from project.simulation import StressTester

stress = StressTester(returns, clean_prices[investable], w_primary, class_map)

historical = stress.historical_scenarios()
print('Historical Stress Test Results:')
print('=' * 100)
# Show portfolio and key asset class columns
cols = [c for c in historical.columns if 'Return' in c or c in ['Days', 'Max_DD_%']]
print(historical[cols].round(2).to_string())

In [ ]:
# Historical stress bar chart
fig, ax = plt.subplots(figsize=(14, 7))

hist_sorted = historical.sort_values('Portfolio_Return_%')
colors = ['red' if v < 0 else 'green' for v in hist_sorted['Portfolio_Return_%']]
hist_sorted['Portfolio_Return_%'].plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_xlabel('Portfolio Return (%)')
ax.set_title(f'Historical Scenario Impact — {primary} Portfolio',
             fontsize=14, fontweight='bold')
ax.axvline(x=0, color='gray', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Multi-strategy historical stress comparison
stress_comparison = {}
for strat in strategies_to_compare:
    s = StressTester(returns, clean_prices[investable], all_weights[strat], class_map)
    h = s.historical_scenarios()
    if 'Portfolio_Return_%' in h.columns:
        stress_comparison[strat] = h['Portfolio_Return_%']

stress_comp_df = pd.DataFrame(stress_comparison)

fig, ax = plt.subplots(figsize=(16, 8))
stress_comp_df.plot(kind='bar', ax=ax, edgecolor='white', width=0.8)
ax.set_ylabel('Portfolio Return (%)')
ax.set_title('Historical Stress Test — Strategy Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc='upper left')
ax.axhline(y=0, color='gray', linewidth=0.5)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 7. Hypothetical Stress Scenarios

In [ ]:
hypo = stress.run_all_hypothetical()

print('Hypothetical Stress Scenarios:')
print('=' * 100)
cols_hypo = [c for c in hypo.columns if c != 'Description']
print(hypo[cols_hypo].round(2).to_string())

In [ ]:
# Waterfall chart for worst scenario
worst_scenario_name = hypo['Portfolio_Impact_%'].idxmin()
scenarios = stress.get_predefined_scenarios()
worst = next(s for s in scenarios if s.name == worst_scenario_name)
worst_result = stress.hypothetical_shock(worst)

fig = ScenarioAnalyzer.plot_stress_waterfall(worst_result)
plt.show()

In [ ]:
# Hypothetical stress for all strategies
hypo_strategies = {}
for scenario in stress.get_predefined_scenarios():
    row = {}
    for strat in strategies_to_compare:
        s = StressTester(returns, clean_prices[investable], all_weights[strat], class_map)
        r = s.hypothetical_shock(scenario)
        row[strat] = r['portfolio_impact_%']
    hypo_strategies[scenario.name] = row

hypo_strat_df = pd.DataFrame(hypo_strategies).T

fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(hypo_strat_df, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Portfolio Impact (%)'})
ax.set_title('Hypothetical Stress Impact — All Strategies (%)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Reverse Stress Test

In [ ]:
reverse = stress.reverse_stress(target_loss=-0.15, n_worst=10)
print('Reverse Stress Test — When would portfolio lose ≥15%?')
print('=' * 70)
if len(reverse) > 0:
    print(reverse.to_string())
else:
    print('No historical periods found with ≥15% portfolio loss.')

## 9. Key Probability Metrics

In [ ]:
# Use bootstrap (most realistic) for final probability assessment
boot = all_sims['Bootstrap']

# Extract metrics for clean display
n_sims = boot['n_sims']
mean_return = boot['mean_return']
median_return = boot['median_return']
std_return = boot['std_return']
prob_positive = boot['prob_positive']
prob_gain_20pct = boot['prob_gain_20pct']
prob_gain_50pct = boot['prob_gain_50pct']
prob_loss_10pct = boot['prob_loss_10pct']
prob_loss_20pct = boot['prob_loss_20pct']
var_5pct = boot['var_5pct']
cvar_5pct = boot['cvar_5pct']
avg_max_drawdown = boot['avg_max_drawdown']
worst_max_drawdown = boot['worst_max_drawdown']
percentiles = boot['percentiles']

print(f"12-Month Forward Outlook — {primary} (Bootstrap MC, {n_sims:,} sims)")
print('=' * 65)
print(f"  Expected return:           {mean_return:.2%}")
print(f"  Median return:             {median_return:.2%}")
print(f"  Std deviation:             {std_return:.2%}")
print(f"")
print(f"  P(positive return):        {prob_positive:.1%}")
print(f"  P(gain > 20%):             {prob_gain_20pct:.1%}")
print(f"  P(gain > 50%):             {prob_gain_50pct:.1%}")
print(f"  P(loss > 10%):             {prob_loss_10pct:.1%}")
print(f"  P(loss > 20%):             {prob_loss_20pct:.1%}")
print(f"")
print(f"  VaR (5%, 1yr):             {var_5pct:.2%}")
print(f"  CVaR (5%, 1yr):            {cvar_5pct:.2%}")
print(f"  Avg max drawdown:          {avg_max_drawdown:.2%}")
print(f"  Worst simulated drawdown:  {worst_max_drawdown:.2%}")
print(f"")
print(f"  Confidence intervals:")
for p in ['p5', 'p25', 'p50', 'p75', 'p95']:
    print(f"    {p}: {percentiles[p]:.2%}")

---

## Key Takeaways from Module 6

1. **Simulation method matters** — Student-t and Bootstrap produce wider tails than Normal, more realistic
2. **GARCH bootstrap** captures volatility clustering — higher probability of prolonged drawdowns
3. **Historical stress tests** validate portfolio resilience against known crisis periods
4. **Hypothetical scenarios** reveal which macro regimes are most dangerous to the portfolio
5. **Risk Parity and HRP** tend to be more resilient across stress scenarios
6. **Crypto exposure** is the dominant source of tail risk in most portfolio strategies

### Forward-Looking Risk Budget
The probability analysis gives actionable guidance:
- If P(loss > 20%) is too high → reduce aggressive allocations
- If worst-case drawdown is unacceptable → add tail hedging or more bonds
- If median return is too low → accept more risk or add alpha signals

---

**QuantVerse Modules 1–6 Complete.**